# SDEV245
## M05 Assignment - OWASP Top 10 Code Fix
### Nathan Thomas

### Broken Acess Control

1. This code does not verify that the userId requested is permissible to be used by the current profile. 

```javascript
app.get('/profile/:userId', (req, res) => {
    User.findById(req.params.userId, (err, user) => {
        if (err) return res.status(500).send(err);
        res.json(user);
    });
});
```

Solution: We can check the user's Session identity against the requested profile.
OWASP A01:2025 Broken Access Control - https://owasp.org/Top10/2025/A01_2025-Broken_Access_Control/

```javascript
app.get('/api/profile/:userId', authenticateToken, (req, res) => {
    const requestedUserId = req.params.userId;
    const authenticatedUserId = req.user.id; // From the secure session profile.

    // Checking if the logged-in user owns this profile
    if (requestedUserId !== authenticatedUserId) {
        return res.status(403).send("Unauthorized: Profile unavailable.");
    }
db.collection('users').findOne({ _id: requestedUserId }, (err, user) => {
    if (err) return res.status(500).send(err);
    if (!user) return res.status(404).send("User not found.");
    });
});
```

2. The function get_account(user_id) grabs the ID directly from the URL path. It does not check to see who is making the request. 

```python
@app.route('/account/<user_id>')
def get_account(user_id):
    user = db.query(User).filter_by(id=user_id).first()
    return jsonify(user.to_dict())
```

Solution: Verify logged in user ID matches the user_id input as string literal. If true, check for user in database and return dictionary if true or error if false.
OWASP A01:2025 Broken Access Control - https://owasp.org/Top10/2025/A01_2025-Broken_Access_Control/

```python
@app.route('/account/<user_id>')
@login_required     # Checks to see if the user is logged in.
def get_account(user_id):
    # Check to verify that logged in account matched requested ID.
    if str(user_id) !=(current_user.id):
        abort(403, description='Access Denied: You cannot view other accounts.')
    
    user = db.query(User).filter_by(id=user_id).first()

    if not user:
        abort(404, description='User not found.')

    return jsonify(user.to_dict())
```

### Cryptographic Failures

3. I believe the vulnerability in this code is that it does not salt the password and it may use a weak hashing algorithm. I have confirmed that MD5 is a weak algorithm in todays sytems.

```java
public String hashPassword(String password) throws NoSuchAlgorithException {
    MessageDigest md = MessageDigest.getInstance('MD5');
    md.update(password.getBytes());
    byte[] digest = md.digest();
    return DatatypeConverter.printHexBinary(digest);
}

```

Solution: Use a more secure cryptographic library and make sure to salt the passwords. We can increase the length of the salt to future proof our security. 
OWASP A04:2025 Cryptographic Failures - https://owasp.org/Top10/2025/A04_2025-Cryptographic_Failures/

```java
import org.mindrot.jbcrypt.Bcrypt;

public String hashPassword(String password) {
    // Generate a random salt automatically
    String hashedPassword = Bcrypt.hashpw(password, Bcrypt.gensalt(12));
}

4. The primary failure happening here is the use sha1 hashing algorithm. This is also another example of unsalted password. 

```python
import haslib
def hash_password(password):
    return hashlib.sha1(password.encode()).hexdigest()
```


Solution: Use a newer and more secure hashing function as well as salt the password. BCrypt is also recommended by OWASP and is designed to be 'memory-hard'.
OWASP A04: 2025 Cryptographic failures - https://owasp.org/Top10/2025/A04_2025-Cryptographic_Failures/

```python
import bcrypt

def hash_password(password):
    # Need to convert the password into bytes
    byte_password = password.encode('utf-8')

    # BCrypt automatically handles the salt.
    hashed = bcrypt.hashpw(byte_password, bcrypt.gensalt())

    return hashed

```

### Injection

5. This code uses string concatenation and uses it as a parameter to pass into SQL commands. 

```java
String username = request.getParameter("username");
String query = "SELECT * FROM users WHERE username = '" + username + "'";
Statement stmt = connection.createStatement();
ResultSet rs = stmt.executeQuery(query);
```

Solution: Use prepared statements with parameterized queries.
OWASP A05:2025 Injection - https://owasp.org/Top10/2025/A05_2025-Injection/

```java
// Using a prepared statement to separate query logic from data.
String username = request.getParameter('username');

// Using a placeholder (?) instead of concatenating the string.
String query = 'SELECT * FROM users WHERE username = ?';

// Now we can prepare the statement.
try (PreparedStatement pstmt = connection.preparStatement(query)) {

    // Bind the user input as a parameter this way the database treats this as pure data
    // and not executable code.
    pstmt.setString(1, username);

    try(ResultSet rs = pstmt.executeQuery()) {
        while (rs.next()) {
            // Do whatever to the result, but safely.
        }
    }
}
```

6. A dynamic query that has not been sanitized is at use here. This allows code to be executed in the interpreter. NoSQL will handle req.query.username as an object even though the program expects a string.

```javascript
app.get('/user', (req, res) => {
    // Directly trusting query parameters can lead to NoSQL injection
    db.collection('users').findOne({ username: req.query.username }, (err, user) => {
        if (err) throw err;
        res.json(user);
    });
});
```

Solution: Sanitize the input or force it to be a specific type like a sting before passing it into the database.
OWASP A05:2025 Injection - https://owasp.org/Top10/2025/A05_2025-Injection/

```javascript
// Sanitzing the input.
const mongoSanitize = require('express-mongo-sanitize');

app.get('/user', (req, res) => {
    // Explicitly cast to a string type or use the sanitizer above);
    const username = String(req.query.username);

    // Now the input is guaranteed to be a literal string, not a malicious object.
    db.collection('users').findOne({ username }, (err, user) => {
        if (err) return res.status(500).json({ error: 'Internal Server Error });
        if (!user) return res.status(404).json({ message: 'User not found' });
        res.json(user);
    });
});
```

### Insecure Design

7. This application does not authenticate the user before allowing you to reset the password after inputing the email. It actually doesn't even validate the email so you could just change the password of any email that exists in the database.

```python
import flask

@app.route('/reset-password', methods=['POST'])
def reset_password():
    email = request.form['email']
    new_password = request.form['new_password']
    user = User.query.filter_by(email=email).first()
    user.password = new_password
    db.session.commit()
    return 'Password reset'
```

Solution: Verify and authenticate credentials and user. Utilize some form of MFA token before proceeding with password reset.
OWASP A06:2025 Insecure Design - https://owasp.org/Top10/2025/A06_2025-Insecure_Design/

```python
import flask

@app.route('/forgot-password', methods=['POST'])
def forgot_password():
    email = request.form['email']
    user = User.query.filter_by(email=email).first()

    if user:
        token = generate_secure_token()
        send_email(user.email, f'Click here to reset: /reset/{token}')

    # Generic response prevents user enumeration
    return 'If an account exists, instructions were sent.'

# Once the secret has been proven and user has been authenticated...
@app.route('/reset-password/<token>', methods=['POST'])
def reset_password(token):
    # We rely on the token that was received instead of the email for security.
    user = User.verify_reset_token(token)
    if not user:
        return 'Invalid or expired link', 403

    user.password = hash_password(request.form['new_password'])
    db.session.commit()
    return 'Success'
```

### Software or Data Integrity Failures

8. The code relies on untrusted sources and content delivery networds. There is no integrity check in place to protect users from malicious activity if the third-party were to be compromised.

```html
<script src='https://cdn.example.com/lib.js'></script>
```

Solution: Add integrity and crossorigin checks to ensure the file has not been tampered with and no cross site shenanigans can occur.
OWASP A08:2025 Software or Data Integrity Failures - https://owasp.org/Top10/2025/A08_2025-Software_or_Data_Integrity_Failures/

```html
<script
    src='https://cdn.example.com/lib.js'
    integrity = 'some-long-hash'
    crossorigin= 'anonymous'>
</script>
```

### Server-Side Request Forgery

9. This code has unvalidated user input in a network request. This allows the attacker to use something like internal URLs to get access to steal information.

```python
import requests

url = input('Enter URL: ')
# The application fetches any URL provided by the user.

response = requests.get(url)
print(response.text)
```

Solution: Whitelisting ('allow lists') ensure the server only talks to known safe endpoints. If possible, restrict input to https to avoid vulnerabilites pertaining to paths like file:// and gopher://. You could disable redirects to prevent attackers from pointing to a 'safe' URL that redirects to something nefarious. 
OWASP  A01:2025 Broken Access Control (added Server-Side Request Forgery) - https://owasp.org/Top10/2025/A01_2025-Broken_Access_Control/

```python
import requests
from urllib.parse import urlparse

# Define allowlist of trusted domains
ALLOWED_DOMAINS = ['://trusted_domain.com', '://example.com']

def secure_fetch(user_url):
    parsed = urlparse(user_url)

    # Check that domain is in allowlist.
    if parsed.netloc not in ALLOWED_DOMAINS:
        return 'Error: Unauthorized domain.'

    # Ensure HTTPS
    if parsed.scheme != 'https':
        return 'Error: Insecure protocol.'

    try:
        # Set timeouts and disable redirects to prevent bypasses
        response = requests.get(user_url, timeout=5, allow_redirects=False)
        return response.text
    except requests.exceptions.RequestException:
        return 'Error: Could not fetch data.'

url = input('Enter URL: ')
print(secure_fetch(url))
```


### Identification and Authentication Failures

10. This code lacks Brute-Force protection. There is no rate limiting or account lockout. The application provides infinite attempts to crack the password.

```java
if (inputPassword.equals(user.getPassword())) {
    // Login success
}
```

Solution: Add rate limiting by tracking failures and blocking the IP or username after a number of failed attempts. Keep responses generic so as not to allude to what was incorrect.
OWASP A07:2025 Authentication Failures - https://owasp.org/Top10/2025/A07_2025-Authentication_Failures/

```java
// apply rate limiting through a pseudo-service.
if (loginThrottler.isBlocked(username)) {
    throw new TooManyRequestsException('Account Temporarily locked. Try again later.');
}

// You should Always use secure hashing.
if (BCrypt.checkpw(inputPassword, user.getPasswordHash())) {
    loginThrottler.resetAttempts(username);
    // Login success
} else {
    // Track failed attempts
    loginThrottler.recordFailure(username);

    // Return generic error to preven user enumeration
    throw new AuthenticationException('Invalid username or password.');
}
```